# 5. Build and deploy



**Purpose:** Prepare Lambda directory, build Docker image, push to ECR, update Lambda, sync frontend to S3. Run once after [3_model_train_shap_ffa.ipynb](3_model_train_shap_ffa.ipynb) and [4_dashboard_visuals.ipynb](4_dashboard_visuals.ipynb).



**Prerequisites:** Notebook 5 (models, metadata, SHAP/FFA combined) and notebook 6 (dashboard visuals) completed. Run from repo root.

In [10]:
# Setup: paths and project root
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "10_risk_dashboard":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "10_risk_dashboard").exists():
    pass
else:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import sync_s3_to_local, check_step_checkpoint_exists, save_step_checkpoint

DASHBOARD_DIR = PROJECT_ROOT / "10_risk_dashboard"
DATA_PREP_DIR = DASHBOARD_DIR / "data_preparation"
DEPLOY_DIR = DASHBOARD_DIR / "deployment"
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

print("PGx Risk Calculator Workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dashboard dir: {DASHBOARD_DIR}")
print(f"Data prep: {DATA_PREP_DIR}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print("=" * 60)

PGx Risk Calculator Workflow
Project root: /home/pgx3874/pgx-analysis
Dashboard dir: /home/pgx3874/pgx-analysis/10_risk_dashboard
Data prep: /home/pgx3874/pgx-analysis/10_risk_dashboard/data_preparation
Data root (NVMe/local): /mnt/nvme


In [ ]:
# Configuration: PGx cohorts and age bands (each cohort has all age bands; from py_helpers.constants)
from py_helpers.constants import REQUIRED_COHORTS

# Input dirs (required for pipeline Step 4–6)
# Cohorts: Step 2 cohort.parquet files (create_model_data reads case/control and target dates from here).
COHORTS_ROOT = DATA_ROOT / "gold" / "cohorts"
# Feature importance: Step 3/3b outputs — cohort_feature_importance.csv and feature_filtering_summary.json per cohort/age_band.
FI_ROOT = DATA_ROOT / "gold" / "feature_importance"
STEP3_OUTPUTS = STEP3B_OUTPUTS = FI_ROOT
# Model data: single canonical location (Step 4 output, Step 5/6 input).
from py_helpers.env_utils import get_model_data_root
MODEL_DATA_ROOT = get_model_data_root()

# Output dirs (Step 6 final model outputs; data prep and Lambda read from these)
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_OUTPUTS_ALT = DATA_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_GOLD = DATA_ROOT / "gold" / "final_model"  # S3 layout: cohort/13-24/*.joblib

print("Cohorts and age bands:")
for cohort, bands in REQUIRED_COHORTS.items():
    print(f"  {cohort}: {bands}")
print("
Input dirs (for Step 4–6):")
print(f"  Cohorts (2):              {COHORTS_ROOT}")
print(f"  Feature importance (3/3b): {FI_ROOT}  (CSVs + feature_filtering_summary.json)")
print(f"  Model data (4; in/out):   {MODEL_DATA_ROOT}")
print("
Output dirs (Step 6):")
print(f"  Project:   {FINAL_MODEL_OUTPUTS}")
print(f"  NVMe:      {FINAL_MODEL_OUTPUTS_ALT}")
print(f"  gold/NVMe: {FINAL_MODEL_GOLD}")

Cohorts and age bands:
  opioid_ed: ['13-24', '25-44', '45-54', '55-64']
  non_opioid_ed: ['65-74', '75-84', '85-94']

Input dirs (for Step 4–6):
  Cohorts (2):              /mnt/nvme/gold/cohorts
  Feature importance (3/3b): /mnt/nvme/gold/feature_importance  (CSVs + feature_filtering_summary.json)
  Model data (4; in/out):   /mnt/nvme/4_model_data

Output dirs (Step 6):
  Project:   /home/pgx3874/pgx-analysis/6_final_model/outputs
  NVMe:      /mnt/nvme/6_final_model/outputs
  gold/NVMe: /mnt/nvme/gold/final_model


## Step 3: Build and deploy risk calculator

Build the Docker image and push to ECR; then update API Gateway/Lambda. Use the deployment script in `10_risk_dashboard/deployment`.

**Order:** Run [4_dashboard_visuals.ipynb](4_dashboard_visuals.ipynb) before this notebook so BupaR, DTW, FP-Growth artifacts exist in S3 before deploy.

### AWS infrastructure configuration

Current deployment (run this cell so Verify infrastructure and Build/deploy use these values):

- **Region:** us-east-1  
- **Account:** 535362115856  
- **ECR:** pgx-risk-calculator  
- **Lambda:** pgx-risk-calculator (role: pgx-lambda-role)  
- **API Gateway:** pgx-risk-calculator (id: cmv0qislq3)  
- **Invoke URL:** https://cmv0qislq3.execute-api.us-east-1.amazonaws.com/prod  
- **S3 dashboard:** s3://jerome-dixon.io/vcu/pgx-risk-calculator/ (Lambda has write; upload HTML here)  
- **Lambda env:** PGX_RESULTS_BUCKET=pgxdatalake, MODEL_CACHE_TTL=3600

In [ ]:
# AWS infrastructure (set env so Verify infrastructure and Build/deploy use these)
import os
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
AWS_ACCOUNT_ID = os.environ.get("AWS_ACCOUNT_ID", "535362115856")
ECR_REPOSITORY = os.environ.get("ECR_REPOSITORY", "pgx-risk-calculator")
ECR_URI = f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/{ECR_REPOSITORY}:latest"
LAMBDA_FUNCTION_NAME = os.environ.get("LAMBDA_FUNCTION_NAME", "pgx-risk-calculator")
LAMBDA_ROLE_NAME = os.environ.get("LAMBDA_ROLE_NAME", "pgx-lambda-role")
LAMBDA_ROLE_ARN = f"arn:aws:iam::{AWS_ACCOUNT_ID}:role/{LAMBDA_ROLE_NAME}"
PGX_API_GATEWAY_NAME = os.environ.get("PGX_API_GATEWAY_NAME", "pgx-risk-calculator")
API_GATEWAY_ID = os.environ.get("API_GATEWAY_ID", "cmv0qislq3")
API_INVOKE_URL = f"https://{API_GATEWAY_ID}.execute-api.{AWS_REGION}.amazonaws.com/prod"
S3_DASHBOARD_BUCKET = os.environ.get("S3_DASHBOARD_BUCKET", "jerome-dixon.io")
S3_DASHBOARD_PREFIX = os.environ.get("S3_DASHBOARD_PREFIX", "vcu/pgx-risk-calculator")
S3_DASHBOARD_PATH = f"s3://{S3_DASHBOARD_BUCKET}/{S3_DASHBOARD_PREFIX}/"

os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_ACCOUNT_ID"] = str(AWS_ACCOUNT_ID)
os.environ["ECR_REPOSITORY"] = ECR_REPOSITORY
os.environ["LAMBDA_FUNCTION_NAME"] = LAMBDA_FUNCTION_NAME
os.environ["LAMBDA_ROLE_NAME"] = LAMBDA_ROLE_NAME
os.environ["PGX_API_GATEWAY_NAME"] = PGX_API_GATEWAY_NAME
os.environ["API_GATEWAY_ID"] = API_GATEWAY_ID
os.environ["S3_DASHBOARD_BUCKET"] = S3_DASHBOARD_BUCKET
os.environ["S3_DASHBOARD_PREFIX"] = S3_DASHBOARD_PREFIX

print(f"Region: {AWS_REGION}  Account: {AWS_ACCOUNT_ID}")
print(f"ECR: {ECR_URI}")
print(f"Lambda: {LAMBDA_FUNCTION_NAME}  Role: {LAMBDA_ROLE_NAME}")
print(f"API: {PGX_API_GATEWAY_NAME} (id: {API_GATEWAY_ID})")
print(f"Invoke URL: {API_INVOKE_URL}")
print(f"S3 dashboard: {S3_DASHBOARD_PATH}")

### Verify infrastructure (Docker, ECR, API Gateway, Lambda)

Verifies Docker, ECR repo, API Gateway, and Lambda function. Run the AWS infrastructure configuration cell first so names/IDs match our deployment.

In [ ]:
# Docker, ECR, API Gateway checks for PGx dashboard
import subprocess
import os

print(f"
{'=' * 80}")
print("Verify infrastructure (Docker, ECR, API Gateway)")
print(f"{'=' * 80}
")

# 1. Docker
print("1. Docker")
print("-" * 40)
try:
    r = subprocess.run(["docker", "ps"], capture_output=True, text=True, timeout=5)
    if r.returncode == 0:
        print("✓ Docker is running")
    else:
        print("⚠ Docker not accessible:", r.stderr.strip() or r.stdout.strip())
        if "permission denied" in (r.stderr or "").lower():
            print("  Linux: sudo usermod -aG docker $USER && newgrp docker")
except FileNotFoundError:
    print("✗ Docker not found — install Docker first")
except Exception as e:
    print(f"⚠ Error: {e}")
print()

# 2. AWS / ECR
print("2. ECR (AWS credentials + repository)")
print("-" * 40)
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
ECR_REPOSITORY = os.environ.get("ECR_REPOSITORY", "pgx-risk-calculator")
try:
    # Get caller identity (proves credentials work)
    r = subprocess.run(
        ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
        capture_output=True, text=True, timeout=10
    )
    if r.returncode != 0:
        print("⚠ AWS CLI not configured or no credentials:", (r.stderr or r.stdout or "").strip())
    else:
        account = r.stdout.strip()
        print(f"✓ AWS identity: account {account}")
    # ECR repository exists?
    r2 = subprocess.run(
        ["aws", "ecr", "describe-repositories", "--repository-names", ECR_REPOSITORY, "--region", AWS_REGION],
        capture_output=True, text=True, timeout=10
    )
    if r2.returncode == 0:
        print(f"✓ ECR repository exists: {ECR_REPOSITORY}")
    else:
        print(f"⚠ ECR repository '{ECR_REPOSITORY}' not found (create it or set ECR_REPOSITORY)")
        print("  docker_build.sh can create it automatically on first push")
except FileNotFoundError:
    print("✗ AWS CLI not found")
except Exception as e:
    print(f"⚠ Error: {e}")
print()

# 3. API Gateway
print("3. API Gateway")
print("-" * 40)
print("Note: API Gateway should already be set up; this step only verifies.")
API_NAME = os.environ.get("PGX_API_GATEWAY_NAME", "pgx-risk-calculator")
try:
    r = subprocess.run(
        ["aws", "apigateway", "get-rest-apis", "--query", f"items[?name=='{API_NAME}'].id", "--output", "text"],
        capture_output=True, text=True, timeout=10
    )
    if r.returncode == 0 and r.stdout.strip():
        api_id = r.stdout.strip().split()[0]
        api_url = f"https://{api_id}.execute-api.{AWS_REGION}.amazonaws.com/prod"
        print(f"✓ API Gateway found: {API_NAME}")
        print(f"  API ID: {api_id}")
        print(f"  Base URL: {api_url}")
        # Optional: test endpoint (e.g. GET /metadata) — use ssl context to avoid CERTIFICATE_VERIFY_FAILED
        try:
            import urllib.request
            import urllib.error
            import ssl
            req = urllib.request.Request(f"{api_url}/metadata", method="GET")
            ctx = ssl.create_default_context()
            try:
                with urllib.request.urlopen(req, timeout=15, context=ctx) as resp:
                    code = resp.getcode()
            except urllib.error.URLError as ue:
                if "CERTIFICATE_VERIFY_FAILED" in str(ue.reason) or "SSL" in str(ue.reason):
                    ctx = ssl._create_unverified_context()
                    with urllib.request.urlopen(req, timeout=15, context=ctx) as resp:
                        code = resp.getcode()
                else:
                    raise
            if code in (200, 301, 302):
                print("  ✓ API is responding")
            else:
                print(f"  ⚠ API returned HTTP {code}")
        except Exception as e:
            err = getattr(e, "code", None) or getattr(e, "status", None) or str(e)[:50]
            if hasattr(e, "code"):
                print(f"  ⚠ API endpoint test: HTTP {e.code} (Lambda may need deployment)")
            else:
                print(f"  ⚠ API endpoint test inconclusive ({err})")
    else:
        print(f"⚠ API Gateway '{API_NAME}' not found or AWS CLI not configured")
        print("  Set up API Gateway and link to Lambda, then re-run this check.")
except Exception as e:
    print(f"⚠ Could not verify API Gateway: {e}")
    print("  Assuming API Gateway is already set up or will be configured after deploy.")
print()

# 4. Lambda
print("4. Lambda")
print("-" * 40)
LAMBDA_FUNCTION_NAME = os.environ.get("LAMBDA_FUNCTION_NAME", "pgx-risk-calculator")
try:
    r = subprocess.run(
        ["aws", "lambda", "get-function", "--function-name", LAMBDA_FUNCTION_NAME, "--region", AWS_REGION],
        capture_output=True, text=True, timeout=10
    )
    if r.returncode == 0:
        print(f"✓ Lambda function exists: {LAMBDA_FUNCTION_NAME}")
    else:
        print(f"⚠ Lambda '{LAMBDA_FUNCTION_NAME}' not found (create via Console/CLI or workflow)")
except Exception as e:
    print(f"⚠ Could not verify Lambda: {e}")

print(f"
{'=' * 80}")


Verify infrastructure (Docker, ECR, API Gateway)

1. Docker
----------------------------------------
✓ Docker is running

2. ECR (AWS credentials + repository)
----------------------------------------
✓ AWS identity: account 535362115856
⚠ ECR repository 'pgx-risk-calculator' not found (create it or set ECR_REPOSITORY)
  docker_build.sh can create it automatically on first push

3. API Gateway
----------------------------------------
Note: API Gateway should already be set up; this step only verifies.
⚠ API Gateway 'pgx-risk-calculator-api' not found or AWS CLI not configured
  Set up API Gateway and link to Lambda, then re-run this check.



### Prepare Models

Package models and feature schemas from `6_final_model/outputs` into `10_risk_dashboard/outputs/models`. **Checkpoint:** step is skipped if S3 checkpoint exists.

In [ ]:
import logging
logger = logging.getLogger(__name__)
if check_step_checkpoint_exists("9_dashboard_models", "all", "all", logger):
    print("Step 3 (prepare models) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "prepare_models.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_models", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

### Prepare CPIC Data

In [ ]:
import logging
logger = logging.getLogger(__name__)
if check_step_checkpoint_exists("9_dashboard_cpic", "all", "all", logger):
    print("Step 3 (prepare cpic data) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "prepare_cpic_data.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_cpic", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

### Prepare Lambda directory

Assemble `lambda_dir` under `10_risk_dashboard` for Docker build (models, metadata, CPIC data). **Prerequisites:** Run Step 1a (metadata), Step 3 (prepare models), and `prepare_cpic_data.py` (in data_preparation) so `outputs/models`, `outputs/metadata`, and `outputs/cpic` exist. The first cell copies those into `lambda_dir`; the second verifies.

In [ ]:
# Prepare first: copy outputs/models, outputs/metadata, outputs/cpic into lambda_dir (required before verify)
r = subprocess.run([sys.executable, "prepare_lambda_dir.py"], cwd=DEPLOY_DIR)
if r.returncode != 0:
    raise SystemExit(r.returncode)

In [ ]:
# Verify lambda_dir has models/, metadata/, data/ (run after prepare cell above)
subprocess.run([sys.executable, "prepare_lambda_dir.py", "--verify-only"], cwd=DEPLOY_DIR, check=True)
print("Lambda directory prepared.")

### Docker Build

In [ ]:
# Variables for Docker build (DASHBOARD_DIR from setup cell)
risk_dashboard_dir = DASHBOARD_DIR
needs_prepare = True  # Set True to prompt for build; False to skip rebuild

print(f"
{'=' * 80}")
print("Step 3: Build and Push Docker Image")
print(f"{'=' * 80}")

docker_script = risk_dashboard_dir / "deployment" / "docker_build.sh"

needs_docker_build = needs_prepare

if docker_script.exists():
    print(f"
Docker build strategy:")
    print(f"  - Lambda directory was {'updated' if needs_prepare else 'unchanged'}")
    print(f"  - Docker image will be {'built' if needs_docker_build else 'skipped (use --force to rebuild)'}")
    print("-" * 80)
    print("
This will:")
    print("  1. Build Docker image with models and dependencies")
    print("  2. Push image to AWS ECR (Elastic Container Registry)")
    print("-" * 80)
    print("
⚠ Note: This requires:")
    print("  - Docker installed and running")
    print("  - AWS CLI configured with ECR permissions")
    print("  - AWS credentials with push access to ECR")
    print("-" * 80)
    
    if needs_docker_build:
        response = input("
Proceed with Docker build? (y/n): ").strip().lower()
    else:
        print("
⏭ Skipping Docker build (Lambda directory unchanged)")
        print("  To force rebuild, run: ./deployment/docker_build.sh")
        response = 'n'
    
    if response == 'y':
        try:
            result = subprocess.run(
                ["bash", str(docker_script)],
                cwd=str(risk_dashboard_dir),
                capture_output=False,
                text=True
            )
            
            if result.returncode == 0:
                print(f"
✓ Docker image built and pushed successfully!")
                print(f"
  Next: Get ECR URI from output above and use it to update Lambda")
            else:
                print(f"
⚠ Docker build exited with code: {result.returncode}")
        except Exception as e:
            print(f"
✗ Error building Docker image: {e}")
            logger.error("Error building Docker image", exc_info=True)
else:
    print(f"
⚠ Docker build script not found: {docker_script}")
    print("  Expected location: 10_risk_dashboard/deployment/docker_build.sh")

print(f"
{'=' * 80}")


Step 3: Build and Push Docker Image

Docker build strategy:
  - Lambda directory was updated
  - Docker image will be built
--------------------------------------------------------------------------------

This will:
  1. Build Docker image with models and dependencies
  2. Push image to AWS ECR (Elastic Container Registry)
--------------------------------------------------------------------------------

⚠ Note: This requires:
  - Docker installed and running
  - AWS CLI configured with ECR permissions
  - AWS credentials with push access to ECR
--------------------------------------------------------------------------------



Proceed with Docker build? (y/n):  y


Building Lambda container image...
Building Docker image...


#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 1.23kB done
#1 DONE 0.0s

#2 [internal] load metadata for public.ecr.aws/lambda/python:3.11
#2 DONE 0.0s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [1/9] FROM public.ecr.aws/lambda/python:3.11
#4 DONE 0.0s

#5 [2/9] RUN yum install -y gcc gcc-c++ cmake make &&     yum clean all &&     rm -rf /var/cache/yum
#5 CACHED

#6 [internal] load build context
#6 transferring context: 4.81kB done
#6 DONE 0.0s

#7 [3/9] COPY backend/requirements.txt /var/task/
#7 DONE 0.0s

#8 [4/9] RUN pip install --no-cache-dir --upgrade pip setuptools wheel &&     pip install --no-cache-dir --prefer-binary -r /var/task/requirements.txt -t /var/task
#8 1.446 Requirement already satisfied: pip in /var/lang/lib/python3.11/site-packages (24.0)
#8 1.524 Collecting pip
#8 1.538   Downloading pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
#8 1.

Logging in to ECR...


WARNING! Your password will be stored unencrypted in /home/pgx3874/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
Checking ECR repository...
{
    "repositories": [
        {
            "repositoryArn": "arn:aws:ecr:us-east-1:535362115856:repository/pgx-risk-calculator",
            "registryId": "535362115856",
            "repositoryName": "pgx-risk-calculator",
            "repositoryUri": "535362115856.dkr.ecr.us-east-1.amazonaws.com/pgx-risk-calculator",
            "createdAt": "2026-02-07T02:22:59.455000+00:00",
            "imageTagMutability": "MUTABLE",
            "imageScanningConfiguration": {
                "scanOnPush": false
            },
            "encryptionConfiguration": {
                "encryptionType": "AES256"
            }
        }
    ]
}
Pushing image to ECR...
The push refers to repository [535362115856.dkr.ecr.us-east-1.amazonaws.com/pgx-risk-calculator]
5f70bf18a086: Preparing
714ae0338a57: Preparing
09ff56cfc450: Preparing
ca315d853674: Preparing
016baaa9b242: Preparing
057aa8f0d171: Preparing
077b8a5e0644: Preparing
bbc96e9e94c1: Preparing
bf8

### Update Lambda

In [ ]:
# Step 4: Update Lambda Function (Idempotent - only if Docker image was updated)
print(f"
{'=' * 80}")
print("Step 4: Update Lambda Function")
print(f"{'=' * 80}")

lambda_function_name = "pgx-risk-calculator"
region = "us-east-1"

# Check if Lambda function exists
lambda_exists = False
try:
    result = subprocess.run(
        ["aws", "lambda", "get-function", "--function-name", lambda_function_name, "--region", region],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        lambda_exists = True
        print(f"
✓ Lambda function exists: {lambda_function_name}")
except:
    print(f"
⚠ Could not check Lambda function status")
    print("  (AWS CLI may not be configured)")

# Get AWS account ID and ECR URI (for update)
account_id = None
ecr_uri = None
try:
    result = subprocess.run(
        ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        account_id = result.stdout.strip()
        ecr_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/pgx-risk-calculator:latest"
        print(f"  AWS Account ID: {account_id}")
        print(f"  ECR URI: {ecr_uri}")
except Exception:
    pass

if lambda_exists and needs_docker_build and account_id and ecr_uri:
    print(f"
Lambda function will be updated with new Docker image")
    print("-" * 80)
    response = input("
Proceed with Lambda update? (y/n): ").strip().lower()
    if response == 'y':
        try:
            result = subprocess.run(
                ["aws", "lambda", "update-function-code", "--function-name", lambda_function_name,
                 "--image-uri", ecr_uri, "--region", region],
                capture_output=False, text=True
            )
            if result.returncode == 0:
                print(f"
✓ Lambda function updated successfully!")
                subprocess.run(["aws", "lambda", "wait", "function-updated", "--function-name", lambda_function_name, "--region", region], capture_output=True)
                print(f"  ✓ Lambda function is ready")
            else:
                print(f"
⚠ Lambda update exited with code: {result.returncode}")
        except Exception as e:
            print(f"
✗ Error updating Lambda: {e}")
    else:
        print("
⏭ Skipping Lambda update")
elif not lambda_exists:
    print(f"
⚠ Lambda function '{lambda_function_name}' does not exist")
    print("  Create it via AWS Console or CLI, then use this cell to update image.")
elif not needs_docker_build:
    print(f"
⏭ Skipping Lambda update (Docker image unchanged)")
else:
    print("
To update Lambda function manually:")
    print("-" * 80)
    print("
1. Get your AWS Account ID:")
    print("   AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)")
    print("
2. Construct ECR URI:")
    print("   ECR_URI=\"${AWS_ACCOUNT_ID}.dkr.ecr.us-east-1.amazonaws.com/pgx-risk-calculator:latest\"")
    print("
3. Update Lambda function:")
    print("   aws lambda update-function-code \\")
    print("       --function-name pgx-risk-calculator \\")
    print("       --image-uri ${ECR_URI} \\")
    print("       --region us-east-1")

print(f"
{'=' * 80}")

### Update HTML/js Front End on S3

In [ ]:
# Step 6: Sync frontend HTML to S3 (local EC2 directory -> S3)
# Dashboard HTML is tracked in git (10_risk_dashboard/frontend/*.html) so local changes deploy here.
print(f"
{'=' * 80}")
print("Step 5: Sync Dashboard Frontend to S3")
print(f"{'=' * 80}")

frontend_dir = risk_dashboard_dir / "frontend"
s3_bucket = os.environ.get("S3_DASHBOARD_BUCKET", "jerome-dixon.io")
s3_prefix = os.environ.get("S3_DASHBOARD_PREFIX", "vcu/pgx-risk-calculator")
s3_uri = f"s3://{s3_bucket}/{s3_prefix}/"

if frontend_dir.exists() and frontend_dir.is_dir():
    print(f"
Source (local): {frontend_dir}")
    print(f"Destination:   {s3_uri}")
    print("  (Only changed or new files are uploaded.)")
    print("-" * 80)
    response = input("
Proceed with S3 sync? (y/n): ").strip().lower()
    if response == 'y':
        try:
            result = subprocess.run(
                ["aws", "s3", "sync", str(frontend_dir), s3_uri,
                 "--region", "us-east-1"],
                capture_output=False, text=True
            )
            if result.returncode == 0:
                print(f"
✓ Frontend synced to S3")
                print(f"  Dashboard URL: https://{s3_bucket}.s3.us-east-1.amazonaws.com/{s3_prefix}/index.html")
                # Upload metrics JSON so Documentation tab loads it same-origin (no API call)
                metrics_path = risk_dashboard_dir / "outputs" / "metadata" / "model_performance_metrics.json"
                if metrics_path.exists():
                    metrics_key = f"{s3_prefix.rstrip('/')}/metadata/model_performance_metrics.json"
                    try:
                        subprocess.run(
                            ["aws", "s3", "cp", str(metrics_path), f"s3://{s3_bucket}/{metrics_key}",
                             "--content-type", "application/json", "--region", "us-east-1"],
                            check=True, capture_output=True, text=True
                        )
                        print(f"  ✓ Metrics uploaded to s3://{s3_bucket}/{metrics_key}")
                    except (subprocess.CalledProcessError, FileNotFoundError) as e:
                        print(f"  ⚠ Metrics upload failed: {e}")
                else:
                    print(f"  ⚠ Metrics file not found: {metrics_path} — run data_preparation/generate_metrics.py to create it.")
                # Upload cohort metadata JSONs so dropdowns load same-origin (no API call)
                metadata_dir = risk_dashboard_dir / "outputs" / "metadata"
                for cohort_name, s3_suffix in [("opioid_ed", "opioid_ed"), ("non_opioid_ed", "non_opioid_ed")]:
                    meta_path = metadata_dir / f"metadata_{cohort_name}.json"
                    if meta_path.exists():
                        meta_key = f"{s3_prefix.rstrip('/')}/metadata/{s3_suffix}.json"
                        try:
                            subprocess.run(
                                ["aws", "s3", "cp", str(meta_path), f"s3://{s3_bucket}/{meta_key}",
                                 "--content-type", "application/json", "--region", "us-east-1"],
                                check=True, capture_output=True, text=True
                            )
                            print(f"  ✓ Metadata uploaded: {meta_key}")
                        except (subprocess.CalledProcessError, FileNotFoundError) as e:
                            print(f"  ⚠ Metadata upload failed ({cohort_name}): {e}")
                if not (metadata_dir / "metadata_opioid_ed.json").exists() or not (metadata_dir / "metadata_non_opioid_ed.json").exists():
                    print(f"  ⚠ Some metadata files missing in {metadata_dir} — run data_preparation/generate_metadata.py --all to create them.")
                # Upload feature importance heatmaps (Step 3a / 2_feature_importance) for dashboard tab
                fi_base = PROJECT_ROOT / "3a_feature_importance" / "outputs"
                fi_prefix = f"{s3_prefix.rstrip('/')}/feature_importance"
                for cohort in ("opioid_ed", "non_opioid_ed"):
                    local_png = fi_base / cohort / "plots" / f"{cohort}_aggregated_fi_heatmap.png"
                    if local_png.exists():
                        s3_key = f"{fi_prefix}/{cohort}/aggregated_fi_heatmap.png"
                        try:
                            subprocess.run(
                                ["aws", "s3", "cp", str(local_png), f"s3://{s3_bucket}/{s3_key}", "--region", "us-east-1"],
                                check=True, capture_output=True, text=True
                            )
                            print(f"  ✓ Feature importance heatmap: {s3_key}")
                        except (subprocess.CalledProcessError, FileNotFoundError) as e:
                            print(f"  ⚠ FI heatmap upload failed ({cohort}): {e}")
                combined_png = fi_base / "plots" / "combined_cohorts_feature_importance_heatmap.png"
                if combined_png.exists():
                    s3_key_combined = f"{fi_prefix}/combined_cohorts_feature_importance_heatmap.png"
                    try:
                        subprocess.run(
                            ["aws", "s3", "cp", str(combined_png), f"s3://{s3_bucket}/{s3_key_combined}", "--region", "us-east-1"],
                            check=True, capture_output=True, text=True
                        )
                        print(f"  ✓ Feature importance heatmap: {s3_key_combined}")
                    except (subprocess.CalledProcessError, FileNotFoundError) as e:
                        print(f"  ⚠ FI combined heatmap upload failed: {e}")
            else:
                print(f"
⚠ S3 sync exited with code: {result.returncode}")
        except Exception as e:
            print(f"
✗ Error syncing to S3: {e}")
    else:
        print("
⏭ Skipping S3 sync")
else:
    print(f"
⚠ Frontend directory not found: {frontend_dir}")
    print("  Expected: 10_risk_dashboard/frontend/ (index.html and assets; tracked in git)")

print(f"
{'=' * 80}")

# Shutdown EC2

In [ ]:
# -------------------------------------------------------------------
# Optional EC2 Auto-Shutdown
# -------------------------------------------------------------------

SHUTDOWN_EC2 = True  # Set to False to disable auto-shutdown

print("=" * 80)
print("Final Step: EC2 Instance Shutdown (Optional)")
print("=" * 80)

if SHUTDOWN_EC2:
    print("\nShutting down EC2 instance...")
    print("-" * 80)

    import subprocess
    import shutil
    import os

    try:
        # Retrieve EC2 instance ID from metadata service
        result = subprocess.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True,
            text=True,
            timeout=5
        )

        instance_id = result.stdout.strip()

        if instance_id:
            print(f"Instance ID: {instance_id}")

            # Locate AWS CLI
            aws_cmd = shutil.which("aws")
            if not aws_cmd:
                for path in [
                    "/usr/local/bin/aws",
                    "/usr/bin/aws",
                    "/home/ec2-user/.local/bin/aws"
                ]:
                    if os.path.exists(path):
                        aws_cmd = path
                        break

            if not aws_cmd:
                print("\nWarning: AWS CLI not found. Cannot stop instance.")
                print("Install AWS CLI or ensure it is in your PATH.")
                logger.warning("AWS CLI not found; cannot stop EC2 instance")
            else:
                shutdown_cmd = [
                    aws_cmd,
                    "ec2",
                    "stop-instances",
                    "--instance-ids",
                    instance_id
                ]

                print(f"Running: {' '.join(shutdown_cmd)}")
                result = subprocess.run(
                    shutdown_cmd,
                    capture_output=True,
                    text=True
                )

                if result.returncode == 0:
                    print("\nEC2 stop command sent successfully.")
                    print("Instance will stop shortly.")
                    print("Note: This is a STOP (not terminate).")
                    logger.info(
                        f"EC2 instance {instance_id} stop command issued"
                    )
                else:
                    print(
                        f"\nWarning: EC2 stop command failed "
                        f"(exit code {result.returncode})"
                    )
                    if result.stderr:
                        print(f"Error: {result.stderr.strip()}")
                    logger.warning(
                        f"EC2 stop command failed: {result.stderr}"
                    )
        else:
            print("\nWarning: Instance ID not found. Skipping shutdown.")
            print("Manual shutdown command:")
            print("  aws ec2 stop-instances --instance-ids <instance-id>")
            logger.warning("EC2 instance ID could not be determined")

    except subprocess.TimeoutExpired:
        print("\nWarning: Timeout contacting EC2 metadata service.")
        logger.warning("Timeout retrieving EC2 instance ID")

    except Exception as e:
        print(f"\nWarning: Error during EC2 shutdown: {e}")
        logger.warning(f"EC2 shutdown exception: {e}")

else:
    print("\nEC2 Auto-Shutdown: DISABLED")
    print("Set SHUTDOWN_EC2 = True to enable it.")

print("\n" + "=" * 80)
print("Workflow Complete!")
print("=" * 80)
